# 2b. Provider catalog, policies and connectors (bearer token)

The Provider publishes what it offers as a DCAT catalog: datasets, their distributions (how the
data is delivered) and ODRL policies (the terms under which it may be used). A connector instance
bound to each distribution tells the Provider how to reach the real backend when a transfer
starts; in this notebook the connectors use a bearer token kept in the Keystore (`/api/v1/keystore/secrets`).

Steps:
1. Onboarding (`setup_auth.ipynb`), guarded.
2. Resolve the Provider's main catalog and data service (`/api/v1/catalog-agent/catalogs/main`, `/data-services/main`).
3. Create a dataset with a PULL distribution (`http+pull`) and a PUSH distribution (`http+push`) (`/api/v1/catalog-agent/datasets`, `/distributions`).
4. Attach an ODRL offer instantiated from a policy template (`/api/v1/catalog-agent/policy-templates`, `/instantiate-odrl-offer`).
5. Attach two ODRL offers written by hand (`/api/v1/catalog-agent/odrl-policies`).
6. Store the backend's API key in the Provider's Keystore (`/api/v1/keystore/secrets`).
7. Register a PULL connector template (`BEARER_TOKEN`), instantiate it on the PULL distribution and inspect it (`/api/v1/connector/*`).
8. Register a PUSH connector template (`BEARER_TOKEN`), instantiate it on the PUSH distribution and inspect it.
9. The Consumer discovers the catalog and dataset over DSP (`/dsp/current/catalog/rpc/setup-catalog-request`, `setup-dataset-request`).

**Runs on its own.** It leaves `target_id`, `policy_id` and `data_space_provider_dsp_endpoint`
ready for the negotiation notebooks, and sets `catalog_setup = "bearer"` so that the transfer
notebooks know which variant is loaded. Each run creates a new dataset. `setup_catalog_policies.ipynb` is the
same flow with connectors without authentication.

## 1. Prerequisites

The guard runs the onboarding notebook only when this kernel has not run it yet. `%run` on a
notebook executes its cells in this same namespace, so the URLs, DIDs and `show` it defines are
available below.

In [1]:
if not globals().get("onboarded"):
    %run ./setup_auth.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet
HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_n

## 2. Main catalog and data service (`/api/v1/catalog-agent/*`)

Every resource of the catalog is JSON-LD in the agent's own context
(`https://w3id.org/eunomia/v1/catalog-context.jsonld`): `@id` is its URN, texts are language maps
and version, visibility and owner travel in headers (`ETag`, `Eunomia-Visibility`). Every Provider
starts with a `main` catalog and a `main` data service. The data service is the
Provider's DSP endpoint; distributions point to it through `accessService`.

In [2]:
catalog = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/catalogs/main", headers=json_header))
catalog_id = catalog["@id"]

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/catalogs/main
{
  "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
  "@id": "urn:catalog:49570cc3-600c-4e1c-9d60-06896f2293df",
  "@type": "Catalog",
  "identifier": "urn:catalog:49570cc3-600c-4e1c-9d60-06896f2293df",
  "issued": "2026-10-26T14:04:31.819Z",
  "participantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ"
}


In [3]:
data_service = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/data-services/main", headers=json_header))
data_service_id = data_service["@id"]

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/data-services/main
{
  "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
  "@id": "urn:data-service:3714c38d-16ed-42fe-a2f0-b5e3e622434c",
  "@type": "DataService",
  "conformsTo": [
    "https://w3id.org/dspace/2025/1/"
  ],
  "endpointURL": "http://127.0.0.1:1200/dsp/current",
  "identifier": "urn:data-service:3714c38d-16ed-42fe-a2f0-b5e3e622434c",
  "inCatalog": "urn:catalog:49570cc3-600c-4e1c-9d60-06896f2293df",
  "issued": "2026-10-26T14:04:31.873Z"
}


## 3. Dataset and distributions (`/api/v1/catalog-agent/datasets`, `/distributions`)

A dataset is the unit of negotiation: a contract agreement always targets a dataset. Its
distributions are the ways to obtain it. The format of a distribution is the `format` a transfer
request asks for later:
- `http+pull`: the Consumer pulls the data from an endpoint the Provider hands over.
- `http+push`: the Provider pushes the data to a sink the Consumer names.

In [4]:
# A Dataset in the catalog's JSON-LD context: texts are language maps, `inCatalog` places it.
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Dataset",
    "title": {"en": "My dataspace dataset", "es": "Mi dataset del espacio de datos"},
    "description": {"en": "Readings shared through the dataspace"},
    "keyword": {"en": ["tutorial", "readings"]},
    "theme": ["http://publications.europa.eu/resource/authority/data-theme/TECH"],
    "inCatalog": catalog_id,
}
dataset = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/datasets", headers=json_header, json=payload))
dataset_id = dataset["@id"]

HTTP 201 POST http://127.0.0.1:1200/api/v1/catalog-agent/datasets
{
  "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
  "@id": "urn:dataset:3e997fd8-0118-44fe-aa02-0e5a8e46c914",
  "@type": "Dataset",
  "description": {
    "en": "Readings shared through the dataspace"
  },
  "identifier": "urn:dataset:3e997fd8-0118-44fe-aa02-0e5a8e46c914",
  "inCatalog": "urn:catalog:49570cc3-600c-4e1c-9d60-06896f2293df",
  "issued": "2026-10-26T20:23:10.757Z",
  "keyword": {
    "en": [
      "tutorial",
      "readings"
    ]
  },
  "theme": [
    "http://publications.europa.eu/resource/authority/data-theme/TECH"
  ],
  "title": {
    "en": "My dataspace dataset",
    "es": "Mi dataset del espacio de datos"
  }
}


In [5]:
# `format` is the DSP transfer format (`dct:format`); `ofDataset` places it under the dataset.
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Distribution",
    "title": {"en": "Distribution of my dataset"},
    "description": {"en": "Pull access to my dataset"},
    "accessService": [data_service_id],
    "ofDataset": dataset_id,
    "format": "http+pull",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_pull_id = distribution["@id"]

HTTP 201 POST http://127.0.0.1:1200/api/v1/catalog-agent/distributions
{
  "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
  "@id": "urn:distribution:1278a52d-53c6-43c6-9909-47dd3b80f834",
  "@type": "Distribution",
  "accessService": [
    "urn:data-service:3714c38d-16ed-42fe-a2f0-b5e3e622434c"
  ],
  "description": {
    "en": "Pull access to my dataset"
  },
  "format": "http+pull",
  "identifier": "urn:distribution:1278a52d-53c6-43c6-9909-47dd3b80f834",
  "issued": "2026-10-26T20:23:17.504Z",
  "ofDataset": "urn:dataset:3e997fd8-0118-44fe-aa02-0e5a8e46c914",
  "title": {
    "en": "Distribution of my dataset"
  }
}


In [6]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Distribution",
    "title": {"en": "Distribution of a streaming dataset"},
    "description": {"en": "Push delivery of my streaming dataset"},
    "accessService": [data_service_id],
    "ofDataset": dataset_id,
    "format": "http+push",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_push_id = distribution["@id"]

HTTP 201 POST http://127.0.0.1:1200/api/v1/catalog-agent/distributions
{
  "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
  "@id": "urn:distribution:554ac9aa-01f0-426d-b2f1-df7d46529027",
  "@type": "Distribution",
  "accessService": [
    "urn:data-service:3714c38d-16ed-42fe-a2f0-b5e3e622434c"
  ],
  "description": {
    "en": "Push delivery of my streaming dataset"
  },
  "format": "http+push",
  "identifier": "urn:distribution:554ac9aa-01f0-426d-b2f1-df7d46529027",
  "issued": "2026-10-26T20:23:27.528Z",
  "ofDataset": "urn:dataset:3e997fd8-0118-44fe-aa02-0e5a8e46c914",
  "title": {
    "en": "Distribution of a streaming dataset"
  }
}


## 4. ODRL offer from a policy template (`/api/v1/catalog-agent/policy-templates`)

Policy templates are parameterised ODRL policies. The agent seeds `policy-1`, whose single
permission constrains `bla` to the value of the `$bla` parameter. Instantiating it attaches the
resulting offer to the dataset.

In [7]:
templates = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/policy-templates", headers=json_header))

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/policy-templates
{
  "items": [
    {
      "author": "",
      "content": {
        "@context": "http://www.w3.org/ns/odrl.jsonld",
        "@type": "Offer",
        "permission": [
          {
            "action": "use",
            "constraint": [
              {
                "leftOperand": "bla",
                "operator": "eq",
                "rightOperand": "$bla"
              }
            ]
          }
        ]
      },
      "date": "2026-10-26T14:04:31.881216Z",
      "description": "Mi example super policy",
      "id": "policy-1",
      "parameters": {
        "$bla": {
          "dataType": "string",
          "restrictions": {},
          "ui": {
            "hidden": false
          }
        }
      },
      "title": "Mi example policy",
      "userId": "system",
      "userRole": "/admin",
      "version": "1.0",
      "visibility": "Public"
    }
  ],
  "total": 1
}


In [8]:
payload = {
    "id": "policy-1",
    "version": "1.0",
    "parameters": {
        "$bla": "did:web:university.upm.es"
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research access for UPM: grants use to did:web:university.upm.es.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer", headers=json_header, json=payload))

HTTP 202 POST http://127.0.0.1:1200/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer
{
  "id": "urn:odrl-policy:c25fb5d9-36a2-4d27-8c2a-3e8f34155779",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private",
  "odrlOffer": {
    "@context": "http://www.w3.org/ns/odrl.jsonld",
    "@type": "Offer",
    "permission": [
      {
        "action": "use",
        "constraint": [
          {
            "leftOperand": "bla",
            "operator": "eq",
            "rightOperand": "did:web:university.upm.es"
          }
        ]
      }
    ]
  },
  "entity": "urn:dataset:3e997fd8-0118-44fe-aa02-0e5a8e46c914",
  "entityType": "Dataset",
  "createdAt": "2026-10-26T20:23:30.507629Z",
  "sourceTemplateId": "policy-1",
  "sourceTemplateVersion": "1.0",
  "instantiationParameters": {
    "$bla": "did:web:university.upm.es"
  },
  "description": "Research access for UPM: grants use to did:web:university.upm.es."
}


## 5. ODRL offers written by hand (`/api/v1/catalog-agent/odrl-policies`)

An offer can also be posted as plain ODRL: permissions, obligations (duties) and prohibitions,
each with optional constraints. The Consumer will see all three offers on the dataset.

In [9]:
payload = {
    "odrlOffer": {
        "@context": "http://www.w3.org/ns/odrl.jsonld",
        "@type": "Offer",
        "permission": [{
            "action": "use",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-01-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-12-31T23:59:59Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "commercial"},
            ],
        }],
        "obligation": [{
            "action": "attribute",
            "constraint": [{"leftOperand": "recipient", "operator": "isA", "rightOperand": "public"}],
        }],
        "prohibition": [{"action": "derive", "constraint": []}],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

HTTP 201 POST http://127.0.0.1:1200/api/v1/catalog-agent/odrl-policies
{
  "id": "urn:odrl-policy:2c7c7ddb-c20a-4f70-856b-a5aedaa40d07",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private",
  "odrlOffer": {
    "@context": "http://www.w3.org/ns/odrl.jsonld",
    "@type": "Offer",
    "obligation": [
      {
        "action": "attribute",
        "constraint": [
          {
            "leftOperand": "recipient",
            "operator": "isA",
            "rightOperand": "public"
          }
        ]
      }
    ],
    "permission": [
      {
        "action": "use",
        "constraint": [
          {
            "leftOperand": "dateTime",
            "operator": "gteq",
            "rightOperand": "2026-01-01T00:00:00Z"
          },
          {
            "leftOperand": "dateTime",
            "operator": "lteq",
            "rightOperand": "2026-12-31T23:59:59Z"
          },
          {
            "leftOperand": "purpose",
            "operator": "isA",
       

In [10]:
payload = {
    "odrlOffer": {
        "@context": "http://www.w3.org/ns/odrl.jsonld",
        "@type": "Offer",
        "permission": [{
            "action": "read",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-03-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-09-01T00:00:00Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "research"},
                {"leftOperand": "count", "operator": "lteq", "rightOperand": "10000"},
            ],
        }],
        "obligation": [],
        "prohibition": [
            {"action": "distribute", "constraint": []},
            {"action": "reproduce", "constraint": []},
        ],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

HTTP 201 POST http://127.0.0.1:1200/api/v1/catalog-agent/odrl-policies
{
  "id": "urn:odrl-policy:f75b4a11-1ac7-45d2-b154-faef26e6a0ff",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private",
  "odrlOffer": {
    "@context": "http://www.w3.org/ns/odrl.jsonld",
    "@type": "Offer",
    "obligation": [],
    "permission": [
      {
        "action": "read",
        "constraint": [
          {
            "leftOperand": "dateTime",
            "operator": "gteq",
            "rightOperand": "2026-03-01T00:00:00Z"
          },
          {
            "leftOperand": "dateTime",
            "operator": "lteq",
            "rightOperand": "2026-09-01T00:00:00Z"
          },
          {
            "leftOperand": "purpose",
            "operator": "isA",
            "rightOperand": "research"
          },
          {
            "leftOperand": "count",
            "operator": "lteq",
            "rightOperand": "10000"
          }
        ]
      }
    ],
    "prohibition": 

## 6. Backend secret in the Keystore (`/api/v1/keystore/secrets`)

The backend behind the connectors expects a bearer token. Instead of writing it into the
connector, it is stored once in the Provider's Keystore under `/ecostars/api_key`; templates refer
to it as `{{__RUNTIME_SECRET_{/ecostars/api_key}__}}` and the agent resolves it on each call.
Set `BACKEND_API_KEY` to store your own token.

A key is created once (`POST /api/v1/keystore/secrets`): on a rerun the cell replaces its value
instead (`PUT /api/v1/keystore/secrets/{key}`), naming the version it replaces (`expectedVersion`),
as the Keystore requires.

In [11]:
secret_key = "/ecostars/api_key"
secret_url = f"{data_space_provider}/api/v1/keystore/secrets"
secret_value = os.environ.get("BACKEND_API_KEY", "demo-backend-api-key")

existing = requests.get(f"{secret_url}{secret_key}", headers=json_header)
if existing.status_code == 404:
    payload = {"@context": "https://w3id.org/eunomia/v1/keystore-context.jsonld", "@type": "Secret", "key": secret_key, "value": secret_value, "description": "Ecostars API key"}
    secret = show(requests.post(secret_url, headers=json_header, json=payload))
else:
    payload = {"@context": "https://w3id.org/eunomia/v1/keystore-context.jsonld", "@type": "Secret", "value": secret_value, "expectedVersion": show(existing)["version"], "description": "Ecostars API key"}
    secret = show(requests.put(f"{secret_url}{secret_key}", headers=json_header, json=payload))

HTTP 201 POST http://127.0.0.1:1200/api/v1/keystore/secrets
{
  "@context": "https://w3id.org/eunomia/v1/keystore-context.jsonld",
  "@type": "Secret",
  "createdAt": "2026-10-26T20:23:33.644395Z",
  "createdBy": "",
  "description": "Ecostars API key",
  "key": "/ecostars/api_key",
  "updatedAt": "2026-10-26T20:23:33.644395Z",
  "updatedBy": "",
  "value": "*****",
  "version": 1
}


## 7. PULL connector (`/api/v1/connector/*`)

A connector template describes how to reach a backend, with `{{__NAME__}}` placeholders; an
instance fills the parameters and binds the template to a distribution. In PULL mode the template
describes the data access the Consumer gets once the transfer starts.

In [12]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorTemplate",
    "authentication": {
        "type": "BEARER_TOKEN",
        "token": {
            "type": "PLAIN",
            "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
        }
    },
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}"
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "ACCESS_URL", "title": "Url", "required": True},
        {"paramType": "VEC<STRING>", "name": "ACCESS_METHODS", "title": "Methods", "required": True},
        {"paramType": "MAP<STRING,STRING>", "name": "ACCESS_HEADERS", "title": "Headers", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/templates
{
  "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
  "@type": "ConnectorTemplate",
  "authentication": {
    "token": {
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}",
      "type": "PLAIN"
    },
    "type": "BEARER_TOKEN"
  },
  "author": "admin",
  "createdAt": "2026-10-26T20:23:41.488727Z",
  "description": null,
  "interaction": {
    "dataAccess": {
      "bodyTemplate": null,
      "headers": "{{__ACCESS_HEADERS__}}",
      "method": "{{__ACCESS_METHODS__}}",
      "protocol": "HTTP",
      "urlTemplate": "{{__ACCESS_URL__}}"
    },
    "mode": "PULL"
  },
  "name": "urn:connector-template:2aace365-d915-4671-8f11-cd4da5a6e58e",
  "parameters": [
    {
      "defaultValue": null,
      "description": null,
      "name": "ACCESS_URL",
      "paramType": "STRING",
      "required": true,
      "title": "Url"
    },
    {
      "defaultValue": null,
      "description": null,
      "na

In [13]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorInstance",
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_pull_id,
    "metadata": {
        "description": "Pull access to the demo backend",
        "ownerId": "tutorial"
    },
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": "http://127.0.0.1:8081",
        "ACCESS_METHODS": ["GET", "HEAD"],
        "ACCESS_HEADERS": {"Content-type": "application/json"}
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
pull_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_pull_id}", headers=json_header))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/instances
{
  "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
  "@type": "ConnectorInstance",
  "authenticationConfig": {
    "token": {
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}",
      "type": "PLAIN"
    },
    "type": "BEARER_TOKEN"
  },
  "author": null,
  "createdAt": "2026-10-26T20:23:42.458744Z",
  "description": null,
  "distributionId": "urn:distribution:1278a52d-53c6-43c6-9909-47dd3b80f834",
  "id": "urn:connector-instance:7046b17d-5efe-417f-8f73-8466f9c6cb26",
  "interaction": {
    "dataAccess": {
      "bodyTemplate": null,
      "headers": {
        "Content-type": "application/json"
      },
      "method": [
        "GET",
        "HEAD"
      ],
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8081"
    },
    "mode": "PULL"
  },
  "name": "urn:connector-template:2aace365-d915-4671-8f11-cd4da5a6e58e",
  "userId": "system",
  "version": "1.0"
}
HTTP 200 GET http:

## 8. PUSH connector (`/api/v1/connector/*`)

In PUSH mode the template says how to subscribe the Provider to the backend when a transfer
starts, and how to unsubscribe when it stops. `SUB_URL` and `UNSUB_URL` are filled by the
instance; the `SYS_*` and `RUNTIME_*` placeholders are filled by the agent at run time (its own
URL, the ingress of the transfer, and the subscription id the backend answered).

In [14]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorTemplate",
    "authentication": {
        "type": "BEARER_TOKEN",
        "token": {
            "type": "PLAIN",
            "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
        }
    },
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}",
                "event_type": "hotel_waste_generated"
            })
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({})
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "SUB_URL", "title": "Subscription URL", "required": True},
        {"paramType": "STRING", "name": "UNSUB_URL", "title": "Unsubscription URL", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/templates
{
  "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
  "@type": "ConnectorTemplate",
  "authentication": {
    "token": {
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}",
      "type": "PLAIN"
    },
    "type": "BEARER_TOKEN"
  },
  "author": "admin",
  "createdAt": "2026-10-26T20:23:52.133594Z",
  "description": null,
  "interaction": {
    "mode": "PUSH",
    "subscribe": {
      "bodyTemplate": "{\"url\": \"{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}",
      "headers": {},
      "method": "POST",
      "protocol": "HTTP",
      "urlTemplate": "{{__SUB_URL__}}"
    },
    "unsubscribe": {
      "bodyTemplate": "{}",
      "headers": {},
      "method": "POST",
      "protocol": "HTTP",
      "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}"
    }
  },
  "name": "urn:connector-template:e3796803-b991-4701-9d8e

In [15]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
    "@type": "ConnectorInstance",
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_push_id,
    "parameters": {
        "SUB_URL": "http://127.0.0.1:8082/subscriptions/subscribe",
        "UNSUB_URL": "http://127.0.0.1:8082/subscriptions/unsubscribe"
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
push_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_push_id}", headers=json_header))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/instances
{
  "@context": "https://w3id.org/eunomia/v1/connector-context.jsonld",
  "@type": "ConnectorInstance",
  "authenticationConfig": {
    "token": {
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}",
      "type": "PLAIN"
    },
    "type": "BEARER_TOKEN"
  },
  "author": null,
  "createdAt": "2026-10-26T20:23:52.777355Z",
  "description": null,
  "distributionId": "urn:distribution:554ac9aa-01f0-426d-b2f1-df7d46529027",
  "id": "urn:connector-instance:f699f3d0-f52f-4012-b92c-9d8a179a9154",
  "interaction": {
    "mode": "PUSH",
    "subscribe": {
      "bodyTemplate": "{\"url\": \"http://host.docker.internal:1200{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}",
      "headers": {},
      "method": "POST",
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8082/subscriptions/subscribe"
    },
    "unsubscribe": {
      "bodyTemplate": "{}",
      "headers": {},
      "method": "

## 9. Catalog and dataset discovery by the Consumer (DSP 6.2.1 & DSP 6.2.2)

The Consumer asks its own agent to fetch the Provider's catalog
(`POST /dsp/current/catalog/rpc/setup-catalog-request`, sending `CatalogRequestMessage`) and
individual dataset (`POST /dsp/current/catalog/rpc/setup-dataset-request`, sending
`DatasetRequestMessage`). From the answer we keep:
- `target_id`: the `@id` of the dataset created above.
- `policy_id`: the `@id` of one of its offers. A negotiation must open on an offer the catalog
  publishes, on the dataset that publishes it.
- `data_space_provider_dsp_endpoint`: the `endpointURL` of the Provider's data service (`service`
  is a list).

In [16]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcCatalogRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "noCache": True,
}
provider_catalog = show(requests.post(f"{data_space_consumer}/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, json=payload))["response"]

HTTP 200 POST http://127.0.0.1:1100/dsp/current/catalog/rpc/setup-catalog-request
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcCatalogResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcCatalogRequestMessage",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "noCache": true
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@id": "urn:catalog:49570cc3-600c-4e1c-9d60-06

In [17]:
dataset_rpc = show(requests.post(
    f"{data_space_consumer}/dsp/current/catalog/rpc/setup-dataset-request",
    headers=json_header,
    json={
        "@context": "https://w3id.org/eunomia/v1/context.jsonld",
        "@type": "RpcDatasetRequestMessage",
        "associatedAgentPeer": data_space_provider_participant_id,
        "dataset": dataset_id,
    },
))["response"]

published = next(d for d in provider_catalog["dataset"] if d["@id"] == dataset_id)
assert dataset_rpc["@id"] == published["@id"]

target_id = published["@id"]
policy_id = published["hasPolicy"][0]["@id"]
data_space_provider_dsp_endpoint = provider_catalog["service"][0]["endpointURL"]

# Marker read by the guards of the negotiation and transfer notebooks
catalog_setup = "bearer"

print(f"Dataset (target):      {target_id}")
print(f"Offer (policy):        {policy_id}")
print(f"Provider DSP address:  {data_space_provider_dsp_endpoint}")

HTTP 200 POST http://127.0.0.1:1100/dsp/current/catalog/rpc/setup-dataset-request
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcCatalogResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcDatasetRequestMessage",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "dataset": "urn:dataset:3e997fd8-0118-44fe-aa02-0e5a8e46c914"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    